<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Traders Autonomos</h2>
            <span style="color:#ff7800;">Una simulacion de trading de acciones para ilustrar agentes autonomos impulsados por herramientas y recursos de servidores MCP.
            </span>
        </td>
    </tr>
</table>

### Semana 6 Dia 4

Y ahora presentamos el proyecto final:


# Traders Autonomos

Una simulacion de trading de acciones, con 4 traders y un investigador, impulsada por varios servidores MCP con herramientas y recursos:

1. Nuestro servidor MCP de cuentas hecho en casa (escrito por nuestro equipo de ingenieria!)
2. Fetch (obtiene paginas web mediante un navegador local sin interfaz grafica)
3. Memoria
4. Busqueda web general con nuestro MCP local `search_server.py`
5. Datos financieros

Y un recurso para leer informacion sobre la cuenta del trader y su estrategia de inversion.

El objetivo del lab de hoy es crear un nuevo modulo de python, `traders.py`, que gestionara un trader individual en nuestro piso de trading.

Vamos a experimentar y explorar en el lab, y luego migraremos el codigo a un modulo de python cuando este listo.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Una vez mas --</h2>
            <span style="color:#ff7800;">Por favor no uses esto para decisiones reales de trading!!
            </span>
        </td>
    </tr>
</table>

In [38]:
import os
import subprocess

import httpx
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio, MCPServerStreamableHttp
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

def get_ollama_base_url():
    """Usa OLLAMA_BASE_URL si existe; si no, detecta el host de Windows desde WSL."""
    configured_url = os.getenv("OLLAMA_BASE_URL") or os.getenv("OLLAMA_HOST")
    if configured_url:
        configured_url = configured_url.rstrip("/")
        return configured_url if configured_url.endswith("/v1") else f"{configured_url}/v1"

    try:
        route = subprocess.check_output(["ip", "route"], text=True, stderr=subprocess.DEVNULL)
        windows_host = next(line.split()[2] for line in route.splitlines() if line.startswith("default "))
        return f"http://{windows_host}:11434/v1"
    except Exception:
        return "http://127.0.0.1:11434/v1"

ollama_url = get_ollama_base_url()
ollama_model_name = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")

print("Usando Ollama en:", ollama_url)
print("Modelo Ollama:", ollama_model_name)

ollama_client = AsyncOpenAI(
    base_url=ollama_url,
    api_key="ollama",
    http_client=httpx.AsyncClient(timeout=120, trust_env=False),
)

ollama_model = OpenAIChatCompletionsModel(
    model=ollama_model_name,
    openai_client=ollama_client,
)

Usando Ollama en: http://172.21.96.1:11434/v1
Modelo Ollama: qwen2.5:7b


### Empecemos reuniendo los parametros MCP para nuestro trader

In [39]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [40]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### Y ahora nuestro investigador

In [41]:
researcher_mcp_server_params = [
    {"command": "uv", "args": ["run", "search_server.py"]},
    {"command": "uvx", "args": ["mcp-server-fetch"]},
]


### Ahora crea un MCPServerStdio para cada servidor

In [42]:
def make_mcp_server(params, timeout=30):
    if params.get("transport") == "streamable_http":
        http_params = {key: value for key, value in params.items() if key != "transport"}
        return MCPServerStreamableHttp(params=http_params, client_session_timeout_seconds=timeout)
    return MCPServerStdio(params=params, client_session_timeout_seconds=timeout)

researcher_mcp_servers = [make_mcp_server(params, timeout=60) for params in researcher_mcp_server_params]
trader_mcp_servers = [make_mcp_server(params, timeout=60) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Ahora creemos un agente investigador para hacer investigacion de mercado

Y convirtamoslo en una herramienta. Recuerda como funciona esto en el OpenAI Agents SDK y cual es la diferencia con los handoffs.

In [43]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="Esta herramienta busca informacion en la web usando web_search y puede leer paginas con fetch_page o fetch. \
                Describe claramente que quieres investigar, incluyendo palabras clave, empresa, ticker, tema o URL si la tienes."
        )

In [44]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""Eres un investigador. Tienes herramientas MCP para buscar en la web y leer paginas concretas.
Cuando necesites encontrar informacion, usa primero web_search con una consulta clara.
Si un resultado parece relevante, usa fetch_page o fetch para leer la pagina concreta.
Segun la solicitud, realiza la investigacion necesaria y responde con tus hallazgos.
Haz varias consultas o lecturas cuando sea util para obtener una vision amplia y luego resume los puntos clave.
Si la busqueda o una pagina fallan, dilo con claridad y no inventes informacion.
La fecha y hora actual es {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model=ollama_model,
        mcp_servers=mcp_servers,
    )
    return researcher

In [45]:
research_question = "Busca en la web informacion reciente sobre Amazon y AMZN. Usa web_search con una consulta clara; si encuentras resultados relevantes, lee uno o dos con fetch_page. Resume los puntos clave y cita URLs."

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Basándome en la información reciente obtenida a través de las fuentes indicadas:

1. **Amazon News**: La web [About Amazon](https://www.aboutamazon.com/news) presenta los principios fundamentales de la empresa: obsessión por el cliente, pasión por la innovación, compromiso con la excelencia operacional y pensamiento a largo plazo.

2. **Google News - Amazon.com**: Se puede acceder a una amplia variedad de noticias y contenido relacionado con Amazon [aquí](https://news.google.com/topics/CAAqIQgKIhtDQkFTRGdvSUwyMHZNRzFuYTJjU0FtVnVLQUFQAQ). Esta fuente ofrece artículos, videos y más actualizaciones.

3. **Bloomberg - Amazon**: Bloomberg cubre las últimas noticias relacionadas con [Amazon](https://www.bloomberg.com/latest/amazon) de una manera detallada e inmediata. 

4. **AMZN News Today | MarketBeat**: [MarketBeat](https://www.marketbeat.com/stocks/NASDAQ/AMZN/news/) actualiza la información relevante sobre AMZN a nivel de contenido y medios confiables.

5. **Yahoo Finance - Amazon Recent Stock News & Headlines**:[Finance Yahoo](https://finance.yahoo.com/quote/AMZN/news/) ofrece las últimas noticias corporativas, anuncios e informes financieros relevantes para la compañía, que son cruciales para los inversores y analistas.

Estas fuentes proporcionan una visión actualizada sobre Amazon y su mercado, abarcando desde la estrategia de empresa hasta las actividades cotidianas en el mercado accionista.

### Nota sobre traces con Ollama

Este lab ahora usa Ollama localmente. Los traces de OpenAI del lab original son opcionales y no hacen falta para esta version.

In [46]:
ed_initial_strategy = "Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-05-18 12:44:08", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.

### Y ahora creemos nuestro agente trader

In [47]:
agent_name = "Ed"

# Usamos servidores MCP para leer recursos
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
Eres un trader que gestiona un portafolio de acciones. Tu nombre es {agent_name} y tu cuenta esta bajo tu nombre, {agent_name}.
Tienes acceso a herramientas para buscar informacion web general, revisar precios de acciones, comprar acciones y vender acciones.
Tu estrategia de inversion para el portafolio es:
{strategy}
Tus posiciones actuales y balance son:
{account_details}
Tienes herramientas para buscar noticias e informacion relevante en la web.
Tienes herramientas para revisar precios de acciones.
Tienes herramientas para comprar y vender acciones.
Tienes herramientas para guardar memoria sobre empresas, investigaciones y razonamientos previos.
Usa estas herramientas para gestionar tu portafolio. Ejecuta operaciones cuando lo consideres apropiado; no esperes instrucciones ni pidas confirmacion.
"""

prompt = """
Usa tus herramientas para tomar decisiones sobre tu portafolio.
Investiga noticias y mercado usando busqueda web cuando necesites informacion actual, toma una decision, ejecuta las operaciones necesarias y responde con un resumen breve de tus acciones.
"""

In [48]:
print(instructions)


Eres un trader que gestiona un portafolio de acciones. Tu nombre es Ed y tu cuenta esta bajo tu nombre, Ed.
Tienes acceso a herramientas para buscar informacion web general, revisar precios de acciones, comprar acciones y vender acciones.
Tu estrategia de inversion para el portafolio es:
Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.
Tus posiciones actuales y balance son:
{"name": "ed", "balance": 10000.0, "strategy": "Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-05-18 12:44:08", 10000.0], ["2026-05-18 12:44:10", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
Tienes herramientas para buscar noticias e informacion relevante en la web.
Tienes herramientas para revisar precios de acciones.
Tienes herramientas para comprar y vender acciones.
Tienes herramientas para guardar mem

### Y ahora ejecutamos nuestro trader

In [63]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model=ollama_model,
)
result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

The trade to buy 10 shares of AAPL at $300.83 has been executed successfully.

Here is the summary of the action:
- **Action**: Bought 10 shares of AAPL at $300.83.
- **New Holdings**: Now holding 10 shares of AAPL.
- **Updated Balance**: The balance now stands at $6,991.6954 after this transaction.
- **Portfolio Value**: The total portfolio value is now $9,993.9954.

This action was taken based on the day trading news and market conditions indicating a potentially bullish environment for tech stocks like AAPL.

Next steps could include monitoring the market and waiting for further signals to make additional trades or adjust positions as needed.

### Nota sobre traces con Ollama

Como esta version usa Ollama localmente, revisa directamente el output del notebook.


In [64]:
# Y veamos los resultados del trading

await read_accounts_resource(agent_name)

'{"name": "ed", "balance": 6991.6954, "strategy": "Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.", "holdings": {"AAPL": 10}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 300.83046, "timestamp": "2026-05-18 12:52:50", "rationale": "Buy into a stable tech stock that shows potential for growth in a bullish market."}], "portfolio_value_time_series": [["2026-05-18 12:44:08", 10000.0], ["2026-05-18 12:44:10", 10000.0], ["2026-05-18 12:47:41", 10000.0], ["2026-05-18 12:47:47", 10000.0], ["2026-05-18 12:49:05", 10000.0], ["2026-05-18 12:49:21", 10000.0], ["2026-05-18 12:52:50", 9993.9954], ["2026-05-18 12:53:44", 9993.9954]], "total_portfolio_value": 9993.9954, "total_profit_loss": -6.0046000000002095}'

### Ahora revisemos el modulo de Python creado a partir de esto:

`mcp_params.py` es donde se especifican los servidores MCP. Veras que traje algunos viejos conocidos: memoria y notificaciones push!

`templates.py` es donde se configuran las instrucciones y mensajes (es decir, los prompts de sistema y de usuario)

`traders.py` une todas las piezas.

Veras que hice algo un poco elegante con codigo como este:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

Esta es simplemente una forma ordenada de combinar nuestros bloques "with" (conocidos como context managers), para no tener que hacer algo feo como esto:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

Pero es equivalente.


In [58]:
from traders import Trader


In [59]:
trader = Trader("Ed")

In [60]:
await trader.run()

Error initializing MCP server: Connection closed


Error ejecutando trader Ed: Connection closed


In [61]:
await read_accounts_resource("Ed")

'{"name": "ed", "balance": 10000.0, "strategy": "Eres un day trader que compra y vende acciones agresivamente segun noticias y condiciones del mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-05-18 12:44:08", 10000.0], ["2026-05-18 12:44:10", 10000.0], ["2026-05-18 12:47:41", 10000.0], ["2026-05-18 12:47:47", 10000.0], ["2026-05-18 12:49:05", 10000.0], ["2026-05-18 12:49:21", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}'

### Nota sobre traces con Ollama

Esta version usa Ollama localmente, asi que revisa directamente los outputs del notebook.

### Cuantas herramientas usamos en total?

In [62]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with make_mcp_server(each_params, timeout=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"Tenemos {len(all_params)} servidores MCP y {count} herramientas")

Error initializing MCP server: Connection closed


McpError: Connection closed